# 04 — Nutrition and training EDA
Cautious descriptive alignment of workout, stored nutrition, body-weight, and strength histories. Timestamp proximity does not establish causation; confounding and sparse observations remain substantial limitations.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from metaform_ds.utils.notebooks import load_notebook_data
data = load_notebook_data(); workouts, nutrition, weight, strength = data.workouts, data.nutrition, data.body_weight, data.strength
print(f'DATA MODE: {data.mode}')
if data.mode == 'SYNTHETIC': print('SYNTHETIC demonstration only.')

In [ ]:
weekly_workouts = workouts.set_index('session_date').resample('W-MON', label='left')['completed'].sum() if not workouts.empty else pd.Series(dtype=float)
if weekly_workouts.empty: print('Insufficient workout history.')
else:
    fig, ax = plt.subplots(); ax.bar(weekly_workouts.index, weekly_workouts.values, width=5); ax.set(title='Completed workout frequency', xlabel='Week starting Monday (UTC)', ylabel='Completed sessions'); fig.autofmt_xdate(); plt.show()

In [ ]:
if nutrition.empty: print('Insufficient nutrition history.')
else:
    fig, ax = plt.subplots(); ax.plot(nutrition['calculated_at'], nutrition['maintenance_calories'], marker='o', label='Maintenance'); ax.plot(nutrition['calculated_at'], nutrition['recommended_calories'], marker='o', label='Recommended'); ax.set(title='Stored calorie targets over time', xlabel='Calculation date (UTC)', ylabel='Calories (kcal/day)'); ax.legend(); fig.autofmt_xdate(); plt.show()

In [ ]:
if weight.empty or nutrition.empty: print('Insufficient timestamp-aligned weight/nutrition history.')
else:
    fig, ax = plt.subplots(); ax.plot(weight['measured_at'], weight['weight_kg'], marker='o', label='Weight (kg)'); ax.set(title='Body weight with nutrition-calculation dates', xlabel='Date (UTC)', ylabel='Weight (kg)'); [ax.axvline(day, alpha=0.25) for day in nutrition['calculated_at']]; ax.legend(); fig.autofmt_xdate(); plt.show()
    print('Vertical markers show calculation timing only; they do not imply an effect on weight.')

In [ ]:
if workouts.empty or strength.empty: print('Insufficient joint workout/strength history.')
else:
    w = workouts.set_index('session_date').resample('W-MON', label='left')['completed'].sum().rename('workouts')
    s = strength.set_index('session_date').resample('W-MON', label='left')['max_weight_kg'].mean().diff().rename('strength_change_kg')
    aligned = pd.concat([w, s], axis=1).dropna()
    if len(aligned) < 3: print('Fewer than 3 aligned weeks; no relationship summary is appropriate.')
    else:
        fig, ax = plt.subplots(); ax.scatter(aligned['workouts'], aligned['strength_change_kg']); ax.set(title='Workout consistency and weekly strength change', xlabel='Completed sessions in week', ylabel='Mean representative-load change (kg)'); plt.show()
        print(f'Descriptive correlation: {aligned.corr().iloc[0,1]:.2f}. This is not causal and is not a prediction.')